In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
# used for human readability of links that include special characters such as à
from urllib.parse import unquote

DATA_DIR = Path("data")

# Load links
links = pd.read_csv(
    DATA_DIR / "links.tsv",
    # files are in .tsv format therefore are seperated by tab spacing not seperated by commas such as .csv
    sep="\t",
    # ignores the comments at the top of the file
    comment="#",
    # changing linkSource to source and linkTarget to target
    names=["source", "target"]
)

# this is just to make the names of links more human readable for example 
# %C3%81ed%C3%A1n_mac_Gabr%C3%A1in becomes Áedán_mac_Gabráin so we are 
# making the À à a human readable form
links["source"] = links["source"].apply(unquote)
links["target"] = links["target"].apply(unquote)


# making sure all links were added
print("Number of links:", links.shape[0])
# checking to see what data looks like
links.head()

Number of links: 119882


,source,target
0,Áedán_mac_Gabráin,Bede
1,Áedán_mac_Gabráin,Columba
2,Áedán_mac_Gabráin,Dál_Riata
3,Áedán_mac_Gabráin,Great_Britain
4,Áedán_mac_Gabráin,Ireland


In [2]:
# get the articles in the same way as with the links
articles = pd.read_csv(
    DATA_DIR / "articles.tsv",
    sep="\t",
    comment="#",
    names=["article"]
)

articles["article"] = articles["article"].apply(unquote)

# making sure all articles were added
print("Number of articles:", articles.shape[0])
# checking to see what data looks like
articles.head()

Number of articles: 4604


,article
0,Áedán_mac_Gabráin
1,Åland
2,Édouard_Manet
3,Éire
4,Óengus_I_of_the_Picts


In [3]:
# this section of code will make the column stocastic matric A used in page rank to rank the articles

# we need the outdegree so that each column is stocasic such that every element in a column is 1/oudegree
# if there is an outdegree as discussed in lecture

# group by source then each group will contain target links
outdegree = links.groupby("source").size()
# used to check with degrees from acutal .tsv file as a sanity check
# print(outdegree)

# list used to make dictionary article_to_idx_dict
article_list = articles["article"].tolist()
# key = article name, value = index number
article_to_idx_dict = {}

for i, article in enumerate(article_list):
    article_to_idx_dict[article] = i

# fills matrix with 0's.  0's wil be changed to 1/oudegree if there is a connection as seen below
n = len(article_list)
A = np.zeros((n, n))

# iterates through each row of links df and replaces the current element 0 of A with 1/outdegree[source]
# where the element of the matrix corresponds to the column (source article) and row (target article)
for _, row in links.iterrows():
    source = row["source"]
    target = row["target"]

    source_idx = article_to_idx_dict[source]
    target_idx = article_to_idx_dict[target]

    # there is a connection from source_idx to target_idx so replace 0 with 1 / outdegree[source]
    A[target_idx, source_idx] = 1 / outdegree[source]

In [4]:
# PageRank power iteration as used in Hw5
def power_iteration_pagerank(A, beta=0.85, tol=1e-8, max_iter=200):
    """
    A: sparse column-stochastic matrix.
       A[i, j] = probability of moving from page j to page i.
    beta: damping factor.
    """

    n = A.shape[0]

    # Start with equal rank for every page
    r = np.ones(n) / n

    # Find columns that sum to 0
    # These are dangling nodes with no outgoing links
    col_sums = np.array(A.sum(axis=0)).flatten()
    dangling = col_sums == 0

    history = []

    for i in range(max_iter):
        # Rank coming from normal links
        link_rank = A @ r

        # Rank from dangling nodes gets spread evenly to all pages
        dangling_rank = r[dangling].sum() / n

        # PageRank update
        new_r = beta * (link_rank + dangling_rank) + ((1 - beta) / n)

        # L1 difference
        diff = np.sum(np.abs(new_r - r))
        history.append(diff)

        r = new_r

        if diff < tol:
            break

    return r, history

In [5]:
articles.head()

,article
0,Áedán_mac_Gabráin
1,Åland
2,Édouard_Manet
3,Éire
4,Óengus_I_of_the_Picts


In [6]:
# check power iteration statistics like we did in Hw5
beta = 0.85

r, hist = power_iteration_pagerank(A, beta)

print("Iterations:", len(hist))
print("Last L1 diff:", hist[-1])
print("Sum of PageRank scores:", r.sum())

Iterations: 35
Last L1 diff: 8.744249859804e-09
Sum of PageRank scores: 1.0


In [7]:
# make a new df with the saved score for viewing and to use later
pagerank_scores = pd.DataFrame({
    "article": article_list,
    # this is the vector r we got from page rank 
    "pagerank": r
})

# make the name of the article easier to read for humans
pagerank_scores["article"] = pagerank_scores["article"].apply(unquote)

# view scores highest to lowest
print(pagerank_scores[["article", "pagerank"]].sort_values("pagerank", ascending=False))
# double checking scores sum to 1
pagerank_scores["pagerank"].sum()

                         article  pagerank
4297               United_States  0.009561
1568                      France  0.006442
1433                      Europe  0.006349
4293              United_Kingdom  0.006245
1389            English_language  0.004873
...                          ...       ...
795   Captain_Marvel_(DC_Comics)  0.000033
789               Cape_Porcupine  0.000033
3144              Palio_di_Siena  0.000033
3129                      PRR_M1  0.000033
1956         History_of_Limerick  0.000033

[4604 rows x 2 columns]


np.float64(1.0)

In [8]:
# creates a dictionary for easier lookups so that we dont have to check the df everytime we need to find
# outgoing links
out_links = {}

for _, row in links.iterrows():
    source = row["source"]
    target = row["target"]

    # add new key if source does not exist
    if source not in out_links:
        out_links[source] = []

    # if exists or after key is added then add the target
    out_links[source].append(target)


# predicts what page should be picked next based on highest ranking PageRank score for outlinks
def predict_pagerank(current_page, out_links, pagerank_scores):
    # get all outlinks from the current page
    outlinks = out_links.get(current_page, [])

    # end if current page is a dead end
    if len(outlinks) == 0:
        return None

    # at first therea are no best pages
    best_page = None
    # guarentees the first outlink will be the best
    best_score = -np.inf

    # check all outlinks to see which one has the highest PageRank score
    for page in outlinks:
        # gets current outlink score or 0 if there is no score
        score = pagerank_scores.get(page, 0)

        # if new score is greater than best_score score becomes best score else move to the next outlink or return
        if score > best_score:
            best_score = score
            best_page = page

    return best_page

In [9]:
# test to see if predict_pagerank works
current_page = links.loc[0, "source"]

prediction = predict_pagerank(current_page, out_links, pagerank_scores)

print("Current page:", current_page)
print("Predicted next page:", prediction)

Current page: Áedán_mac_Gabráin
Predicted next page: Bede


In [17]:
# gets the finished paths so we can test page rank aginst actual games
paths_finished = pd.read_csv(
    DATA_DIR / "paths_finished.tsv",
    sep="\t",
    comment="#",
    header=None,
    names=["hashed_ip", "timestamp", "duration_sec", "path", "rating"]
)

# makes the link names more human readable
paths_finished["path"] = paths_finished["path"].apply(unquote)

# paths_finished.head()
# print to see how the games are parsed
paths_finished["path"]

0        14th_century;15th_century;16th_century;Pacific...
1        14th_century;Europe;Africa;Atlantic_slave_trad...
2        14th_century;Niger;Nigeria;British_Empire;Slav...
3           14th_century;Renaissance;Ancient_Greece;Greece
4        14th_century;Italy;Roman_Catholic_Church;HIV;R...
                               ...                        
51313                     Yagan;Ancient_Egypt;Civilization
51314    Yagan;Folklore;Brothers_Grimm;<;19th_century;C...
51315    Yagan;Australia;England;France;United_States;T...
51316    Yarralumla,_Australian_Capital_Territory;Austr...
51317                              Ziad_Jarrah;Germany;Jew
Name: path, Length: 51318, dtype: object

In [18]:
# make dictionary for every articles PageRank score for easy look up later
pagerank_scores = {}

for i, article in enumerate(article_list):
    pagerank_scores[article] = r[i]
pagerank_scores["Yagan"]

np.float64(3.7120737835014586e-05)

In [28]:
results = []

# loops through all rows of completed games in paths_finished df
for path_id, row in paths_finished.iterrows():
    path_string = row["path"]
    
    # makes "page1;page2;page3" format into ["page1", "page2", "page3"].
    pages = path_string.split(";")

    # wasn't sure how to handle games where players moved back pages
    # I will have to figure this out later
    if "<" in pages:
        continue

    target_page = pages[-1]
    path_length = len(pages) - 1

    # each click in the path is one prediction problem.
    for step in range(len(pages) - 1):
        current_page = pages[step]
        actual_next_page = pages[step + 1]

        # gets all outlinks from the current page so that we can let PageRank decide what link to chose
        candidates = out_links[current_page]

        # needed becasue parts of the .tsv file were added that shouldnt be there
        # also makes sense that if the player picked the page then it should be in the list of outlinks
        if actual_next_page not in candidates:
            continue

        # page rank decides what page to choose
        predicted_next_page = predict_pagerank(current_page, out_links, pagerank_scores)

        # was the actual page selected the same as the one by page rank?
        correct = predicted_next_page == actual_next_page

        # rank all possible next clicks by their PageRank score this will be used so we can see what rank
        # the page was by the player for later evaluation
        ranked_candidates = sorted(candidates, key=lambda page: pagerank_scores.get(page, 0), reverse=True)

        # the actual rank of the page that the player selected
        actual_rank = ranked_candidates.index(actual_next_page) + 1

        # one row of evaluation results for this player selection
        results.append({
            "path_id": path_id,
            "step": step,
            "path_length": path_length,
            "current_page": current_page,
            "target_page": target_page,
            "actual_next_page": actual_next_page,
            "pagerank_prediction": predicted_next_page,
            "correct": correct,
            "actual_rank": actual_rank,
            "num_candidates": len(candidates)})

pagerank_eval = pd.DataFrame(results)

pagerank_eval.head()

,path_id,step,path_length,current_page,target_page,actual_next_page,pagerank_prediction,correct,actual_rank,num_candidates
0,0,0,8,14th_century,African_slave_trade,15th_century,France,False,13,31
1,0,1,8,15th_century,African_slave_trade,16th_century,France,False,15,57
2,0,2,8,16th_century,African_slave_trade,Pacific_Ocean,France,False,21,91
3,0,3,8,Pacific_Ocean,African_slave_trade,Atlantic_Ocean,United_States,False,19,87
4,0,4,8,Atlantic_Ocean,African_slave_trade,Accra,United_States,False,117,125


In [31]:
percent = pagerank_eval["correct"].mean()

print("PageRank chose player page with a percentage of :", percent*100)
average_rank_user = pagerank_eval["actual_rank"].mean()
print("PageRank chose page with an average rank of:", average_rank_user)

PageRank chose player page with a percentage of : 7.276146463206647
PageRank chose player page with a percentage of : 27.58999904971967
